# 6주차 Basic RAG 질의응답 실습 - 학생용

**GitHub 실습 데이터 → Document → Chunking → Embedding → Chroma → Retrieval → LLM → Answer**

교수자 GitHub의 `6w/data` 폴더에 있는 PDF, DOCX, TXT를 Colab으로 가져와 Basic RAG를 구현합니다.

`_____` 부분을 채운 뒤 위에서부터 순서대로 실행하세요.


## 1. 라이브러리 설치

RAG 실습에 필요한 LangChain, Chroma, 임베딩, PDF/DOCX 로더 관련 라이브러리를 설치합니다.


In [ ]:
# 1. 라이브러리 설치
!pip -q install -U langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers pypdf docx2txt transformers accelerate sentencepiece


**확인:** 설치가 오류 없이 완료되면 다음 단계로 진행합니다.


## 2. GitHub 실습 데이터 가져오기

교수자의 GitHub 저장소에 있는 `6w/data` 실습 문서를 가져오는 방법을 세 가지로 확인합니다.

- **2-1. 교수자 GitHub → 학생 GitHub**: 학생 자신의 GitHub에 Fork하여 저장소를 복사합니다.
- **2-2. 교수자 GitHub → 학생 Colab**: 현재 실행 중인 Colab의 `/content` 폴더로 저장소를 가져옵니다.
- **2-3. 교수자 GitHub → 학생 Google Drive**: GitHub의 실습 데이터를 학생 자신의 Google Drive에 저장합니다.

각 방법은 복사 후 **저장 위치와 파일 목록을 확인**합니다.


### 2-1. 교수자 GitHub → 학생 GitHub로 복사

GitHub 계정 간 저장소 복사는 **Fork**를 사용합니다.

1. 교수자 저장소 `hjlee-crypto/2026-ai-sw-convergence`에 접속합니다.
2. 오른쪽 위 **Fork**를 선택합니다.
3. 학생 자신의 GitHub 계정을 선택하여 저장소를 생성합니다.
4. Fork가 완료되면 아래 코드에 자신의 GitHub ID를 입력하여 저장소와 `6w/data` 경로를 확인합니다.


In [ ]:
# 2-1. 학생 GitHub에 Fork한 저장소 경로 확인

# 자신의 GitHub ID 입력
STUDENT_GITHUB_ID = "본인의_GitHub_ID"

# 학생 GitHub에 Fork된 저장소 주소
STUDENT_REPO_URL = f"https://github.com/{STUDENT_GITHUB_ID}/2026-ai-sw-convergence"

# 학생 GitHub 저장소의 6주차 실습 데이터 위치
STUDENT_DATA_URL = STUDENT_REPO_URL + "/tree/main/6w/data"

print("학생 GitHub 저장소:")
print(STUDENT_REPO_URL)

print("\n학생 GitHub의 6주차 데이터 경로:")
print(STUDENT_DATA_URL)


**확인:** 출력된 주소를 브라우저에서 열어 학생 자신의 GitHub 저장소와 `6w/data` 폴더가 존재하는지 확인합니다.


### 2-2. 교수자 GitHub → 학생 Colab으로 복사

교수자의 GitHub 저장소를 **여러분이 현재 실행 중인 Colab 환경의 `/content` 폴더**로 가져옵니다.

`REPO_PATH`는 교수자 GitHub 저장소가 현재 Colab에 복사될 폴더 위치이고,  
`DATA_PATH`는 그중 이번 6주차 RAG 실습에서 사용할 `6w/data` 폴더 위치입니다.


In [ ]:
# 2-2. 교수자 GitHub 저장소를 현재 Colab으로 가져오기

import os

# 교수자가 제공한 GitHub 저장소 주소
GITHUB_URL = "https://github.com/hjlee-crypto/2026-ai-sw-convergence.git"

# 교수자의 GitHub 저장소가 여러분의 현재 Colab 실행 환경에 복사될 폴더 위치
REPO_PATH = "/content/2026-ai-sw-convergence"

# 복사된 교수자 저장소에서 이번 6주차 실습 데이터를 읽어올 폴더 위치
DATA_PATH = REPO_PATH + "/6w/data"

# 저장소가 아직 Colab에 없다면 교수자 GitHub 저장소를 복사
if not os.path.exists(REPO_PATH):
    !git clone -q {GITHUB_URL} {REPO_PATH}

# 복사된 위치와 실습 파일 확인
print("Colab에 복사된 저장소 경로:")
print(REPO_PATH)

print("\n6주차 실습 데이터 경로:")
print(DATA_PATH)

print("\n가져온 실습 파일:")
for filename in sorted(os.listdir(DATA_PATH)):
    print("-", filename)


**확인:** 교수자 GitHub 저장소가 현재 Colab에 복사되고, `/content/2026-ai-sw-convergence/6w/data` 경로와 실습 파일 목록이 출력되는지 확인합니다.


### 2-3. 교수자 GitHub → 학생 Google Drive에 저장

먼저 학생 자신의 Google Drive를 Colab에 연결합니다.

그다음 2-2에서 Colab으로 가져온 교수자 GitHub의 `6w/data` 파일을 학생 Google Drive의 `MyDrive/6w_RAG_data` 폴더에 복사합니다.


In [ ]:
# 2-3. 학생 Google Drive 연결

from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# 교수자 GitHub의 실습 데이터를 학생 Google Drive에 저장

import os
import shutil

# 2-2에서 Colab으로 가져온 교수자 GitHub의 6주차 데이터 위치
SOURCE_PATH = DATA_PATH

# 학생 자신의 Google Drive에 저장할 폴더 위치
DRIVE_DATA_PATH = "/content/drive/MyDrive/6w_RAG_data"

# Google Drive에 실습 데이터 폴더 생성
os.makedirs(DRIVE_DATA_PATH, exist_ok=True)

# 교수자 GitHub의 실습 파일을 학생 Google Drive로 복사
for filename in os.listdir(SOURCE_PATH):
    source_file = os.path.join(SOURCE_PATH, filename)
    destination_file = os.path.join(DRIVE_DATA_PATH, filename)

    if os.path.isfile(source_file):
        shutil.copy2(source_file, destination_file)

# Google Drive 저장 위치와 파일 확인
print("학생 Google Drive 저장 경로:")
print(DRIVE_DATA_PATH)

print("\nGoogle Drive에 저장된 실습 파일:")
for filename in sorted(os.listdir(DRIVE_DATA_PATH)):
    print("-", filename)


**확인:** 학생 자신의 Google Drive에서 `내 드라이브 → 6w_RAG_data` 폴더를 열어 PDF, DOCX, TXT 실습 파일이 저장되었는지 확인합니다.

> 이후 Basic RAG 실습에서는 **2-2에서 설정한 `DATA_PATH`**를 사용하여 교수자 GitHub에서 Colab으로 가져온 문서를 읽습니다.


## 3. PDF, DOCX, TXT 문서를 Document로 불러오기

`DATA_PATH`에 있는 파일을 확장자에 맞는 Loader로 읽습니다.

- PDF → `PyPDFLoader`
- DOCX → `Docx2txtLoader`
- TXT → `TextLoader`

파일 형식이 달라도 LangChain에서는 모두 `Document` 객체로 처리합니다.


In [ ]:
# 3. PDF, DOCX, TXT 문서 불러오기

from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader
)

documents = []

for filename in sorted(os.listdir(DATA_PATH)):
    file_path = os.path.join(DATA_PATH, filename)

    if filename.lower().endswith(".pdf"):
        loader = PyPDFLoader(file_path)

    elif filename.lower().endswith(".docx"):
        loader = Docx2txtLoader(file_path)

    elif filename.lower().endswith(".txt"):
        loader = TextLoader(file_path, encoding="utf-8")

    else:
        continue

    documents.extend(loader.load())

print("불러온 Document 수:", len(documents))


**확인:** `Document 수`가 0보다 커야 합니다. PDF는 페이지별로 여러 Document가 생성될 수 있습니다.


## 4. Document 확인

`page_content`에는 문서의 텍스트가, `metadata`에는 파일명과 페이지 등의 출처 정보가 들어 있는지 확인합니다.


In [ ]:
# 4. Document 내용과 파일명 확인

for i, doc in enumerate(documents, 1):
    filename = os.path.basename(doc.metadata.get("source", "알 수 없음"))

    print(f"\n--- Document {i} ---")
    print("파일명:", filename)
    print("Metadata:", doc.metadata)
    print("내용:")
    print(doc.page_content[:700])


**확인:** 문서 본문과 `source`, `page` 등의 Metadata가 정상적으로 출력되는지 확인합니다.


## 5. Chunking

긴 문서를 한 번에 검색하지 않고 의미 있는 작은 단위인 Chunk로 나눕니다. 일부 내용을 겹치게 하여 문맥이 끊기는 것을 줄입니다.


In [ ]:
# 5. Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print("Document 수:", len(documents))
print("Chunk 수:", len(chunks))


**확인:** 원본 Document보다 더 작은 단위의 Chunk가 생성됩니다.


## 6. Chunk와 Metadata 확인

분할된 Chunk의 내용과 원본 파일, 페이지 정보가 유지되는지 확인합니다.


In [ ]:
# 6. Chunk와 출처 확인
for i, chunk in enumerate(chunks[:8], 1):
    print(f"\n--- Chunk {i} ---")
    print("출처:", chunk.metadata.get("source"))
    print("페이지:", chunk.metadata.get("page", "-"))
    print(chunk.page_content[:500])


**확인:** Chunk가 나뉘어도 원본 파일과 페이지 정보가 함께 유지됩니다.


## 7. Embedding

문장과 질문을 의미를 나타내는 숫자 벡터로 변환합니다. 비슷한 의미의 문장은 벡터 공간에서도 가까워집니다.


In [ ]:
# 7. Embedding
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
vector = embedding_model.embed_query("인턴십 모집인원은 몇 명인가요?")
print("Embedding 차원:", len(vector))
print(vector[:10])


**확인:** 자연어 질문이 고정 길이의 숫자 벡터로 변환됩니다.


## 8. Chroma Vector Store 생성

각 Chunk의 벡터와 원문, Metadata를 Chroma에 저장하여 의미 기반 검색이 가능하도록 합니다.


In [ ]:
# 8. Chroma Vector Store
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="week6_basic_rag"
)
print("Vector Store 생성 완료")


**확인:** 이제 질문과 의미가 가까운 Chunk를 검색할 준비가 완료되었습니다.


## 9. 유사도 검색

질문을 벡터로 변환하고 Vector Store에서 의미가 가장 가까운 Chunk를 검색합니다.


In [ ]:
# 9. 유사도 검색
question = "품종 탐지에는 어떤 모델을 사용하나요?"

results = vectorstore.similarity_search(
    question,
    k=3
)

for i, doc in enumerate(results, 1):
    print(f"\n[검색 결과 {i}]")
    print("출처:", doc.metadata.get("source"))
    print(doc.page_content[:600])


**확인:** 질문과 관련된 공지 내용이 상위 검색 결과에 포함되는지 원문과 비교합니다.


## 10. 여러 질문으로 검색 확인

서로 다른 질문을 입력하여 필요한 문서가 실제로 검색되는지 확인합니다. 아직 LLM은 사용하지 않습니다.


In [ ]:
# 10. 여러 질문으로 Retrieval 확인
questions = [
    "제머나이소프트 인턴십에서 AI 관련 어떤 경험을 우대하나요?",
    "IPP 참가 신청서 제출기한은 언제인가요?",
    "IPP 이력서 제출기한은 언제인가요?"
]
for question in questions:
    print("\n질문:", question)
    results = vectorstore.similarity_search(question, k=3)
    for doc in results:
        print("출처:", doc.metadata.get("source"))
        print(doc.page_content[:450])


**확인:** 질문이 달라지면 검색되는 Chunk와 출처도 달라지는지 확인합니다.


## 11. Retriever

Vector Store의 검색 기능을 Retriever 형태로 만들어 이후 RAG 과정에서 쉽게 사용할 수 있도록 합니다.


In [ ]:
# 11. Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("IPP 현장실습 지원 자격요건은 무엇인가요?")
for doc in docs:
    print(doc.metadata.get("source"))
    print(doc.page_content[:500])


**확인:** Retriever도 Vector Store와 동일하게 관련 문서를 반환합니다.


## 12. 오픈소스 LLM 준비

검색된 문서를 읽고 최종 답변을 생성할 오픈소스 언어 모델을 준비합니다.


In [ ]:
# 12. 오픈소스 LLM
from transformers import pipeline
generator = pipeline(
    "text2text-generation",
    model="paust/pko-t5-base",
    max_new_tokens=200
)
print("LLM 준비 완료")


**확인:** `LLM 준비 완료`가 출력되면 답변 생성 모델을 사용할 수 있습니다.


## 13. Basic RAG 구성

질문으로 관련 문서를 검색하고, 검색된 내용을 Context로 만들어 질문과 함께 LLM에 전달합니다.


In [ ]:
# 13. Basic RAG 함수
def basic_rag(question, k=3):
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""아래 참고 문서만 근거로 질문에 한국어로 간단하고 정확하게 답하세요.
문서에서 확인할 수 없는 내용은 추측하지 말고
'제공된 문서에서 확인할 수 없습니다.'라고 답하세요.

[참고 문서]
{context}

[질문]
{question}

[답변]
"""
    answer = generator(prompt, max_new_tokens=160)[0]["generated_text"]
    return answer, docs


**핵심 흐름:** `질문 → Retrieval → Context 구성 → Prompt → LLM → Answer`가 하나의 함수로 연결되었습니다.


## 14. RAG 질의응답 실행

실제 질문을 입력하여 Retrieval과 Generation이 연결된 전체 RAG 흐름을 실행합니다.


In [ ]:
# 14. RAG 질의응답
question = "제머나이소프트 인턴십 모집인원은 몇 명인가요?"
answer, docs = basic_rag(question)
print("질문:", question)
print("답변:", answer)
print("\n검색 출처")
for doc in docs:
    print("-", doc.metadata.get("source"), "/ page:", doc.metadata.get("page", "-"))


**확인:** 답변뿐 아니라 어떤 원본 파일과 페이지가 검색되었는지도 함께 확인합니다.


## 15. 여러 질문으로 RAG 확인

여러 질문을 연속으로 실행하여 질문에 따라 검색 문서와 최종 답변이 달라지는지 확인합니다.


In [ ]:
# 15. 여러 문서를 대상으로 질문 실행
questions = [
    "Oxford-IIIT Pet 데이터는 몇 개 품종을 다루나요?",
    "YOLO 학습 가중치 파일명은 무엇인가요?",
    "탐지와 분류의 차이는 무엇인가요?",
    "IPP 참가 신청서 제출기한은 언제인가요?"
]

for question in questions:
    answer, docs = basic_rag(question)

    print("\n질문:", question)
    print("답변:", answer)


**확인:** 답변의 근거가 실제 검색된 문서 내용과 일치하는지 비교합니다.


## 16. 문서에 없는 질문 테스트

업로드한 문서에 없는 내용을 질문하여 RAG가 근거 없는 답변을 생성하는지 확인합니다.


In [ ]:
# 16. 문서에 없는 질문 테스트
question = "에펠탑의 높이는 몇 미터인가요?"

answer, docs = basic_rag(question)

print("질문:", question)
print("답변:", answer)

for doc in docs:
    print("-", doc.metadata.get("source"))


**생각해보기:** Vector Store는 가장 가까운 문서를 반환할 수 있으므로, 검색 결과가 있다고 해서 반드시 정답의 근거가 있는 것은 아닙니다.


## 17. 자유 질문

마지막으로 원하는 질문을 직접 입력하여 업로드한 문서를 대상으로 질의응답을 수행합니다.


In [ ]:
# 17. 원하는 질문 직접 입력
question = input("문서에 질문하세요: ")
answer, docs = basic_rag(question)
print("\n답변:", answer)
print("\n검색 출처")
for doc in docs:
    print("-", doc.metadata.get("source"), "/ page:", doc.metadata.get("page", "-"))


**실습 완료:** 실제 문서 파일을 외부 지식으로 사용하는 Basic RAG의 전체 과정을 완료했습니다.
